# Linopy benchmark suite — guide

This notebook is the canonical documentation for the internal benchmark
suite under `benchmarks/`. CI executes it end-to-end on every PR via
`python -m benchmarks notebook`, so anything written here stays runnable.

Two complementary surfaces:

- **README** (`benchmarks/README.md`) — install / size-tier reference / metrics.
- **CLI help** — `python -m benchmarks --help` is the source of truth for
  command flags. This notebook embeds that output live (see the *Running*
  section) so it stays in sync with the actual implementation.

What this notebook covers: the suite's architecture (registry × phases),
how to use the registry from your own code, how to run / interpret
benchmarks, and how to extend the suite.

## Architecture

Two halves:

1. **The model registry** (`benchmarks/registry.py`) — every benchmark
   model is a `ModelSpec` declaring how to build it, its sizes, the
   features it exercises, the phases it can drive, and the size tiers.
   Models self-register at import.

2. **The phase tests** (`test_build.py`, `test_matrices.py`, …) — one
   pytest file per phase. Each iterates the registry via
   `iter_params(phase)` so adding a model to the registry automatically
   extends every applicable phase test.

A typer CLI (`benchmarks/cli.py`) wraps pytest plus introspection and
memory snapshots.

## What each phase measures

| Phase             | Test file                         | Measures                                                     |
| ----------------- | --------------------------------- | ------------------------------------------------------------ |
| `build`           | `test_build.py`                   | constructing variables / expressions / constraints / objective |
| `matrices`        | `test_matrices.py`                | `A`, `b`, `c`, bounds, labels, `Q` for QP                    |
| `lp_write`        | `test_lp_write.py`                | `model.to_file(...)` — LP / MPS serialization                |
| `netcdf`          | `test_netcdf.py`                  | `to_netcdf` and `read_netcdf` round-trip                     |
| `solver_handoff`  | `test_solver_handoff.py`          | `lp.io.to_highspy` / `to_gurobipy` / `to_mosek` / `to_xpress` |
| (PyPSA scenario)  | `test_pypsa_carbon_management.py` | `set_names` / `freeze_constraints` variants — *optional, needs `pypsa`* |

Out of scope: solver algorithm runtime (i.e. `Solver.solve()`),
cross-solver ranking, nonlinear suites.

`pypsa` is an optional benchmark dependency — install it (`pip install pypsa`)
if you want the `pypsa_scigrid` registry spec and the carbon-management
scenario to run; otherwise both skip gracefully.

## The registry

### Import

Single entry point: `from benchmarks import REGISTRY` plus whichever
feature / phase tags you need for filtering. The cell below also defines
a `show_help(...)` helper used later to embed live `--help` output.

In [ ]:
# The benchmark suite isn't shipped in the linopy wheel — it lives in-tree.
# Walk up from cwd to find the repo root and put it on sys.path so the
# import resolves whether jupyter was launched from the repo root, the
# notebooks directory, or anywhere in between.
import os
import subprocess
import sys
from pathlib import Path

_p = Path.cwd()
while _p != _p.parent:
    if (_p / "benchmarks" / "registry.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
    _p = _p.parent

from benchmarks import (  # noqa: E402
    INTEGER,
    QUADRATIC,
    REGISTRY,
    TO_GUROBIPY,
    filter_by,
    get,
)


def show_help(*subcommand: str) -> None:
    """
    Shell out to ``python -m benchmarks ... --help`` and print the output.

    Subprocesses don't inherit ``sys.path`` so we forward the repo root via
    PYTHONPATH. ``NO_COLOR=1`` makes rich emit plain text suitable for the
    notebook's text-output mechanism.
    """
    cmd = [sys.executable, "-m", "benchmarks", *subcommand, "--help"]
    env = {**os.environ, "PYTHONPATH": str(_p), "NO_COLOR": "1"}
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=env,
        check=True,
    )
    print(result.stdout)


print(f"{len(REGISTRY)} models registered: {sorted(REGISTRY)}")

### Look up by name

`REGISTRY[name]` returns a `ModelSpec` (frozen dataclass). Evaluating it
in Jupyter renders an attribute table via `_repr_html_`; `__repr__` gives
a one-line summary in scripts and `pytest -v` output.

In [ ]:
spec = REGISTRY["basic"]
spec

`.build(size)` constructs and returns a `linopy.Model`. Models pick up
their own repr from linopy:

In [ ]:
spec.build(50)

`get("name")` is a functional equivalent to `REGISTRY[name]` — handy when
you don't want to import `REGISTRY` directly.

In [ ]:
assert get("basic") is REGISTRY["basic"]

### Iterate

`REGISTRY.values()` yields every spec. Useful for sweeping your own
regression logic or any operation that should hold across every model.

In [ ]:
list(REGISTRY.values())

### Filter

`filter_by(has_feature=...)` and `filter_by(has_phase=...)` narrow to
specs that declare a given feature or phase. Tags exported from
`benchmarks`: `CONTINUOUS`, `BINARY`, `INTEGER`, `QUADRATIC`, `SOS`,
`PIECEWISE`, `MASKED`, plus `BUILD`, `MATRICES`, `LP_WRITE`, `NETCDF`,
`SOLVER_BUILD`, `TO_HIGHSPY`, `TO_GUROBIPY`, `TO_MOSEK`, `TO_XPRESS`.

In [ ]:
filter_by(has_feature=QUADRATIC)

In [ ]:
filter_by(has_feature=INTEGER)

Useful for solver-specific tests — find every spec that declares the
Gurobi handoff phase (i.e. claims Gurobi can ingest it):

In [ ]:
filter_by(has_phase=TO_GUROBIPY)

## Reuse patterns

### Parametrize your own pytest

The same pattern the suite uses internally — `iter_params(phase)`
flattens `(spec, size)` pairs for one phase, and `param_ids(...)` builds
stable pytest IDs:

```python
import pytest
from benchmarks import BUILD, iter_params, param_ids

_PARAMS = iter_params(BUILD)

@pytest.mark.parametrize("spec,size", _PARAMS, ids=param_ids(_PARAMS))
def test_my_invariant(spec, size):
    m = spec.build(size)
    # ... assertion that should hold for every model
```

### Spot-profile memory

`tracemalloc` is a fast, in-process spot check. For real measurement
(peak RSS, separate process per benchmark) use
`python -m benchmarks memory save / compare`, which routes through
pytest-memray.

In [ ]:
import tracemalloc  # noqa: E402

tracemalloc.start()
m = REGISTRY["sparse_network"].build(100)
_current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"sparse_network n=100: peak allocation ≈ {peak / 1e6:.1f} MB")
print(
    f"  {m.variables.nvars} scalar variables, {m.constraints.ncons} scalar constraints"
)

## Running

All commands are subcommands of `python -m benchmarks`. The CLI is
self-documenting; the cells below embed its `--help` output live. If you
change a flag in `benchmarks/cli.py`, re-running this notebook updates
the documentation automatically.

Three size tiers gate cost. Each spec declares its own thresholds:

| Mode         | Sizes included            | Wall-clock |
| ------------ | ------------------------- | ---------- |
| `smoke`      | `size <= quick_threshold` | ~18 s      |
| `run`        | `size <= long_threshold`  | ~45 s      |
| `run --long` | all sizes                 | ~2 min     |

Top-level command menu:

In [ ]:
show_help()

### Timing snapshots

`run` is the main timing entry point. Its flags:

In [ ]:
show_help("run")

Use `--json` to save pytest-benchmark's output for later diffing — the
JSON includes per-test min / median / IQR over multiple iterations:

```bash
# baseline (e.g. on master)
python -m benchmarks run --json .benchmarks/master.json

# candidate (e.g. on your branch)
python -m benchmarks run --json .benchmarks/my-feature.json

# pytest-benchmark ships its own diff tool:
pytest-benchmark compare .benchmarks/master.json .benchmarks/my-feature.json
```

IQR is the metric to trust on short runs — it stays stable across noise
in a way that min / mean can't.

### Memory snapshots

`memory save` runs the build phase under pytest-memray in a separate
process per benchmark (so peak RSS doesn't accumulate across tests) and
writes JSON. `memory compare` diffs two snapshots:

In [ ]:
show_help("memory")

```bash
python -m benchmarks memory save master
python -m benchmarks memory save "$(git rev-parse --short HEAD)"
python -m benchmarks memory compare master "$(git rev-parse --short HEAD)"
```

Memory is build-only because later phases include build allocations and
attribution becomes unreliable.

### Cross-version sweep

`sweep` bootstraps perf history against published linopy releases. For
each version it builds a fresh uv venv, installs the lockfile + that
linopy, runs the suite, and saves a JSON snapshot.

In [ ]:
show_help("sweep")

```bash
python -m benchmarks sweep 0.5.0 0.6.0 0.7.0 \
    --output-dir .benchmarks/sweep
```

The current (repo-tip) benchmark code runs against each linopy version,
so the measurement layer is constant. Specs whose APIs aren't present in
older linopy (currently `sos` and `piecewise`) skip themselves gracefully
via the `_API_AVAILABLE` gate at registration time.

## Extending

### Adding a new model

1. Drop `benchmarks/models/<name>.py` with a `build_<name>(size) -> Model`.
2. Build a `ModelSpec` and call `register(...)` at module scope. Declare
   realistic `quick_threshold` / `long_threshold` so the smoke stays fast.
3. Add an import in `benchmarks/models/__init__.py` so registration fires.

Every phase test picks the spec up automatically through
`iter_params(phase)`.

### Regenerating the lockfile

After bumping pins in `pyproject.toml`'s `[benchmarks]` extra, regenerate
`benchmarks/requirements.lock`:

```bash
uv pip compile pyproject.toml \
    --extra benchmarks --extra dev --extra solvers \
    --no-emit-package linopy \
    -o benchmarks/requirements.lock
```

The `--no-emit-package linopy` exclusion is critical — without it, the
lockfile pins linopy itself and `sweep` can't vary it.